# 00 · 盤點 SFT 資產（DPO 前的準備）

**目的**：在動手做 DPO 之前，先把 SFT 階段留下的東西全部印出來看清楚：

1. SFT 資料集 `yoyu001/grid-dispatch-notes-zh` 的欄位、切分、樣本
2. 哪些樣本是「違反 SLA → 拒絕」，拒絕時模型輸出長什麼樣子
3. QLoRA adapter `yoyu001/grid-dispatch-A-instruct-qlora-v1` 的 LoRA 設定
4. 評估集（test / 未見過的說法 / 20 題人工題）還在不在

**不需要 GPU**，Settings → Accelerator 選 None 即可（省額度）。
跑完後，把最後一格產生的 `sft_assets_summary.json` 內容貼回給 Claude。

**事前準備**：Kaggle 右上 Add-ons → Secrets → 新增 `HF_TOKEN`（Hugging Face 的 read token），並勾選給這本 notebook 使用。

## 1. 安裝套件

- `datasets`：讀 Hugging Face 上的資料集
- `huggingface_hub`：列檔案、下載單一檔案（例如 `adapter_config.json`）
- `-q` = quiet，只印錯誤；`-U` = 升級到最新版

In [ ]:
!pip install -q -U datasets huggingface_hub

## 2. 用 Kaggle Secrets 登入 Hugging Face

你的資料集是私人的，所以要帶 token。**不要把 token 直接寫在程式裡**——notebook 推到 GitHub 時會一起被公開。
`UserSecretsClient` 會在執行當下從 Kaggle 的保險箱取出 token，程式碼裡看不到。

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)          # 之後 datasets / hub 的呼叫都會自動帶這個 token
print("登入身分:", whoami()["name"])

## 3. 設定要盤點的 repo 名稱

集中放在最上面，之後若改名只要改這裡。
`repo_type` 告訴 Hub 這是 model 還是 dataset（同名時兩者是不同的 repo）。

In [ ]:
DATASET_ID = "yoyu001/grid-dispatch-notes-zh"
ADAPTER_ID = "yoyu001/grid-dispatch-A-instruct-qlora-v1"
MERGED_ID  = "yoyu001/grid-dispatch-A-instruct-merged-v1"

summary = {}   # 最後會存成 json 給 Claude 看

## 4. 列出每個 repo 裡有哪些檔案

先看「有什麼」，再決定「讀什麼」。如果出現 404 / 401，代表名稱錯了或 token 沒有權限，訊息會被記到 summary 裡。

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for rid, rtype in [(DATASET_ID, "dataset"), (ADAPTER_ID, "model"), (MERGED_ID, "model")]:
    try:
        files = api.list_repo_files(rid, repo_type=rtype)
        summary[f"files::{rid}"] = files
        print(f"\n[{rtype}] {rid}  ({len(files)} files)")
        for f in files:
            print("  -", f)
    except Exception as e:
        summary[f"files::{rid}"] = f"ERROR: {e}"
        print(f"\n[{rtype}] {rid}  -> 讀取失敗: {e}")

## 5. 讀入 SFT 資料集，看切分與欄位

- `load_dataset` 回傳 `DatasetDict`，每個 key 是一個 split（例如 `train`、`test`）
- `.features` 是欄位的型別定義，等同資料表的 schema
- 這一步就能回答「SFT 時的評估資料還找得到嗎？」——如果有 `test`、`ood`、`human` 之類的 split 就是它

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID)
print(ds)

summary["splits"] = {name: {"num_rows": d.num_rows, "features": str(d.features)} for name, d in ds.items()}

## 6. 每個 split 印 3 筆完整樣本

`json.dumps(..., ensure_ascii=False)` 讓中文正常顯示（否則會變成 `\u96fb` 這種跳脫字元）；`indent=2` 讓巢狀結構好讀。

In [ ]:
import json

def show(row, max_chars=3000):
    s = json.dumps(row, ensure_ascii=False, indent=2)
    return s if len(s) <= max_chars else s[:max_chars] + "\n...(截斷)"

summary["samples"] = {}
for name, d in ds.items():
    print(f"\n==================== split: {name} ====================")
    picks = [d[i] for i in range(min(3, d.num_rows))]
    summary["samples"][name] = picks
    for i, row in enumerate(picks):
        print(f"\n--- {name}[{i}] ---")
        print(show(row))

## 7. 找出「違反 SLA → 拒絕」的樣本

我們還不知道資料用哪個欄位標示拒絕，所以先用兩種方式找：

1. **欄位名稱**：有沒有叫 `label`、`category`、`is_reject`、`rule` 之類的欄位 → 直接統計數量
2. **文字關鍵字**：只搜尋 **assistant 的回答**，找「拒絕 / reject / SLA / refuse / violation」

> ⚠️ 第一版搜尋了整筆資料，結果 3954/3954 全部命中——因為每筆的 system prompt 都寫著 refuse。
> 教訓：關鍵字搜尋要避開每筆都重複的固定文字（system prompt、模板），否則會被污染。

DPO 最有價值的 pair 往往就在這裡：*chosen* = 正確拒絕，*rejected* = 模型乖乖照做違規指令。

In [ ]:
import re
from collections import Counter

KEYWORDS = re.compile(r"拒絕|拒绝|違反|违反|SLA|reject|refus|violat|denied|不允許|不予", re.I)
first_split = list(ds.keys())[0]
d0 = ds[first_split]

# (1) 看起來像分類標籤的欄位
label_like = [c for c in d0.column_names if re.search(r"label|category|type|reject|rule|sla|intent|status", c, re.I)]
summary["label_like_columns"] = {}
for c in label_like:
    try:
        cnt = Counter(map(str, d0[c])).most_common(20)
        summary["label_like_columns"][c] = cnt
        print(f"欄位 {c} 的值分布：", cnt)
    except Exception as e:
        print(c, "無法統計：", e)

# (2) 關鍵字搜尋：只看 assistant 回答，避開 system prompt
def assistant_text(row):
    msgs = row.get("messages") or []
    return " ".join(m["content"] for m in msgs if m.get("role") == "assistant") or json.dumps(row, ensure_ascii=False)

hits = [i for i in range(d0.num_rows) if KEYWORDS.search(assistant_text(d0[i]))]
print(f"\n[{first_split}] 含拒絕/SLA 關鍵字的樣本: {len(hits)} / {d0.num_rows}")
summary["reject_keyword_hits"] = {"split": first_split, "count": len(hits), "total": d0.num_rows}
summary["reject_samples"] = [d0[i] for i in hits[:5]]
for i in hits[:5]:
    print(f"\n--- {first_split}[{i}] ---")
    print(show(d0[i]))

## 8. 讀 LoRA adapter 設定

`adapter_config.json` 記錄 SFT 時用的 LoRA 超參數。DPO 若要「接著同一個 adapter 繼續訓練」，就必須知道這些值：

| 參數 | 意思 | 白話 |
|---|---|---|
| `r` | LoRA 秩（rank） | 小矩陣的寬度，越大可學的東西越多、記憶體越多 |
| `lora_alpha` | 縮放係數 | 實際影響力 ≈ alpha / r |
| `lora_dropout` | dropout 機率 | 防止過擬合 |
| `target_modules` | 掛 LoRA 的層 | 例如 q_proj/k_proj/v_proj/o_proj/gate/up/down |
| `base_model_name_or_path` | 底模 | 應該是 Qwen/Qwen3-4B-Instruct-2507 |

In [ ]:
from huggingface_hub import hf_hub_download

try:
    p = hf_hub_download(ADAPTER_ID, "adapter_config.json")
    cfg = json.load(open(p))
    keep = ["base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules",
            "bias", "task_type", "use_dora", "use_rslora", "modules_to_save"]
    summary["adapter_config"] = {k: cfg.get(k) for k in keep}
    print(json.dumps(summary["adapter_config"], ensure_ascii=False, indent=2))
except Exception as e:
    summary["adapter_config"] = f"ERROR: {e}"
    print("讀不到 adapter_config.json:", e)

## 9. 讀資料集與 adapter 的 README（資料怎麼產生、SLA 規則可能寫在這）

In [ ]:
for rid, rtype in [(DATASET_ID, "dataset"), (ADAPTER_ID, "model")]:
    try:
        p = hf_hub_download(rid, "README.md", repo_type=rtype)
        text = open(p, encoding="utf-8").read()
        summary[f"readme::{rid}"] = text[:6000]
        print(f"\n========== {rid} README ==========\n", text[:6000])
    except Exception as e:
        summary[f"readme::{rid}"] = f"ERROR: {e}"
        print(rid, "沒有 README 或讀不到:", e)

## 10. 存檔

輸出到 `/kaggle/working/`，右側 Output 面板就能下載。把 `sft_assets_summary.json` 的內容貼回給 Claude（或上傳檔案），下一步就能寫 DPO 資料產生程式。

In [ ]:
out = "/kaggle/working/sft_assets_summary.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
print("已存檔:", out)
print("大小約", round(len(json.dumps(summary, ensure_ascii=False, default=str)) / 1024, 1), "KB")